# Forget gate vs. learning rate in a Titans test-time memory

A Titans memory exposes two learned controls as it reads: the **forget gate** `alpha_t` (how long a value
survives) and the **learning rate** `theta_t` (how hard each token is written). This notebook trains the
identical Memory-as-Context transformer on each corpus and measures **both** off the trained model:
`alpha` (chunk-level) -> horizon `tau = -1/ln(1-alpha)`, and `theta` (token-level), reporting each control's
mean and its within-sequence coefficient of variation. It sweeps seeds (for error bars) and widths.

All corpora are quantile-binned to 256 bins so marginal entropy is pinned at log(256)=5.545 nats and only
temporal structure differs. A mid-run abort guards against memorisation. Runs on a single A100 (Colab).

## 1 - Setup

In [ ]:
!pip install -q titans-pytorch datasets scipy pandas
import torch, numpy as np, time, json, os
from titans_pytorch import MemoryAsContextTransformer, MemoryMLP
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_BINS = 256; UNIFORM = float(np.log(N_BINS))
print('device:', DEVICE, '| uniform baseline (nats):', round(UNIFORM, 3))

In [ ]:
# --- helpers: quantise, marginal entropy, lag-1 mutual information ---
def quantize(series, n_bins=N_BINS, split=0.9):
    s = np.asarray(series, dtype=np.float64); s = s[np.isfinite(s)]
    k = int(len(s) * split)
    edges = np.quantile(s[:k], np.linspace(0, 1, n_bins + 1)[1:-1])
    return (torch.from_numpy(np.digitize(s[:k], edges)).long(),
            torch.from_numpy(np.digitize(s[k:], edges)).long())

def marginal_entropy(tok):
    c = np.bincount(tok.numpy(), minlength=N_BINS).astype(float); p = c / c.sum(); p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def lag1_mi(tok, coarse=16):
    x = (tok.numpy().astype(int) * coarse) // N_BINS
    J = np.histogram2d(x[:-1], x[1:], bins=[coarse, coarse])[0]; J = J / J.sum()
    px, py = J.sum(1, keepdims=True), J.sum(0, keepdims=True); m = J > 0
    return float((J[m] * np.log(J[m] / (px @ py)[m])).sum())

In [ ]:
# --- model (identical builder to the paper) + the two measurements ---
def build(dim, depth=4, seg=8):
    return MemoryAsContextTransformer(
        num_tokens=256, dim=dim, depth=depth, segment_len=32, neural_memory_segment_len=seg,
        num_persist_mem_tokens=4, num_longterm_mem_tokens=4, neural_memory_layers=(2,),
        dim_head=64 if dim >= 256 else 32, heads=6 if dim >= 256 else 2,
        neural_memory_model=MemoryMLP(dim, depth=2, expansion_factor=4.),
        neural_memory_kwargs=dict(dim_head=dim, heads=1), use_flex_attn=False)

def sample_batch(data, seq_len, batch_size):
    st = torch.randint(0, data.size(0) - seq_len - 1, (batch_size,))
    return torch.stack([data[s: s + seq_len + 1] for s in st])

@torch.no_grad()
def val_loss(model, data_val, seq_len, batch, device, n=20):
    model.eval()
    v = float(np.mean([model(sample_batch(data_val, seq_len, batch).to(device), return_loss=True).item()
                       for _ in range(n)]))
    model.train(); return v

@torch.no_grad()
def collect_memory_signals(model, passage, device):
    # one forward pass; capture the forget gate alpha (post-sigmoid, chunk-level) and the
    # token-level learning rate theta = adaptive_step_transform(to_adaptive_step)
    mem = next(g[4] for g in model.layers if g[4] is not None)
    buf = {'alpha': [], 'theta': []}
    h1 = mem.to_decay_factor.register_forward_hook(
        lambda m, i, o: buf['alpha'].append(o.sigmoid().detach().float().reshape(-1).cpu()))
    h2 = mem.to_adaptive_step.register_forward_hook(
        lambda m, i, o: buf['theta'].append(mem.adaptive_step_transform(o).detach().float().reshape(-1).cpu()))
    model(passage.unsqueeze(0).to(device), return_cache=True)
    h1.remove(); h2.remove()
    return torch.cat(buf['alpha']).numpy(), torch.cat(buf['theta']).numpy()

## 2 - Corpora

AR(1) is generated at scale (20M points) so memorisation is unavailable. `electricity_15min` streams from
the public Chronos datasets. **StarLightCurves** is the real UCR corpus: upload `StarLightCurves_TRAIN.tsv`
and `StarLightCurves_TEST.tsv` to `/content` (or point `UCR_ROOT` at a Drive folder). The UCR archive is
public but password-gated, so it cannot be auto-downloaded.

In [ ]:
def ar1_big(n, phi, seed=0):
    rng = np.random.default_rng(seed); e = rng.standard_normal(n)
    if phi == 0.0: return e
    from scipy.signal import lfilter
    return lfilter([1.0], [1.0, -phi], e)

def stream_chronos(config, target_tokens, repo='autogluon/chronos_datasets'):
    from datasets import load_dataset
    ds = load_dataset(repo, config, split='train', streaming=True)
    parts, total, col = [], 0, None
    for row in ds:
        if col is None:
            best, bl = None, 0
            for k, v in row.items():
                if isinstance(v, (list, np.ndarray)) and len(v) > bl:
                    try: float(v[0]); best, bl = k, len(v)
                    except (TypeError, ValueError, IndexError): pass
            col = best
        s = np.asarray(row[col], dtype=np.float64); s = s[np.isfinite(s)]
        if len(s) < 200: continue
        parts.append(s); total += len(s)
        if total >= target_tokens: break
    return np.concatenate(parts)

def load_ucr(root, name='StarLightCurves'):
    import pandas as pd
    def rd(p): return pd.read_csv(p, sep='\t', header=None).values[:, 1:]
    paths = []
    for suf in ('_TRAIN.tsv', '_TEST.tsv'):
        for cand in (os.path.join(root, name, name + suf), os.path.join(root, name + suf)):
            if os.path.exists(cand): paths.append(cand); break
    if not paths: raise FileNotFoundError(name + ' TSVs not found under ' + root)
    return np.concatenate([rd(p).reshape(-1) for p in paths])

## 3 - Config

In [ ]:
DIM = 384            # width; sweep e.g. [64, 384, 512] by re-running with different DIM
STEPS = 6000
SEQ_LEN = 256; BATCH = 8; LR = 2e-4; SEG = 8; PASSAGE = 1024
SEEDS = [0, 1, 2]    # multiple seeds -> error bars
TOKEN_TARGET = 20_000_000
UCR_ROOT = '/content'
RESULTS = f'lr_horizon_dim{DIM}.json'
print(f'dim {DIM} | steps {STEPS} | seeds {SEEDS}')

## 4 - Train and measure both controls (with a mid-run overfit abort)

For each (seed, corpus): train, then read `alpha` and `theta` over the held-out passage. The abort fires if
the train-validation gap exceeds 1.0 (the memorisation signature); white noise correctly sits at uniform.

In [ ]:
rows = json.load(open(RESULTS)) if os.path.exists(RESULTS) else []
done = {(r['corpus'], r['seed']) for r in rows if 'seed' in r}
for seed in SEEDS:
    corpora = {
        'AR_phi0.00':        (lambda s=seed: ar1_big(TOKEN_TARGET, 0.00, s)),
        'AR_phi0.90':        (lambda s=seed: ar1_big(TOKEN_TARGET, 0.90, s)),
        'electricity_15min': (lambda: stream_chronos('electricity_15min', TOKEN_TARGET)),
        'StarLightCurves':   (lambda: load_ucr(UCR_ROOT, 'StarLightCurves')),
    }
    for name, loader in corpora.items():
        if (name, seed) in done: print('skip', name, seed); continue
        try: tr, va = quantize(loader())
        except Exception as e:
            print('load_fail', name, seed, str(e)[:120]); continue
        ent, mi = marginal_entropy(tr), lag1_mi(tr)
        torch.manual_seed(seed); np.random.seed(seed)
        model = build(DIM).to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=LR)
        passage = va[:PASSAGE]; t0 = time.time(); aborted = False
        model.train()
        for step in range(STEPS):
            loss = model(sample_batch(tr, SEQ_LEN, BATCH).to(DEVICE), return_loss=True)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5); opt.step()
            if step in (1500, 3000, 4500):
                v = val_loss(model, va, SEQ_LEN, BATCH, DEVICE, n=10); gap = v - loss.item()
                print(f'  {name} s{seed} step {step} train {loss.item():.3f} val {v:.3f} gap {gap:+.3f}')
                if gap > 1.0:
                    print(f'  !! ABORT {name} s{seed} (gap {gap:.2f}) -- memorising'); aborted = True; break
        if aborted:
            rows.append({'corpus': name, 'seed': seed, 'dim': DIM, 'status': 'aborted_overfit'})
        else:
            alpha, theta = collect_memory_signals(model, passage, DEVICE)
            a = float(alpha.mean()); tau = -1.0 / np.log(1.0 - a)
            rows.append({'corpus': name, 'seed': seed, 'dim': DIM, 'status': 'ok',
                         'lag1_mi': mi, 'val_loss': val_loss(model, va, SEQ_LEN, BATCH, DEVICE),
                         'alpha_mean': a, 'alpha_cv': float(alpha.std()/(abs(a)+1e-12)),
                         'theta_mean': float(theta.mean()),
                         'theta_cv': float(theta.std()/(abs(theta.mean())+1e-12)),
                         'tau_positions': float(tau*SEG), 'minutes': (time.time()-t0)/60})
            r = rows[-1]
            print(f'  -> {name} s{seed}: alpha {a:.4f} | theta {r["theta_mean"]:.4g} '
                  f'(cv {r["theta_cv"]:.3f}) | tau {r["tau_positions"]:.1f} | val {r["val_loss"]:.3f}')
        json.dump(rows, open(RESULTS, 'w'), indent=2)
        del model, opt
        if DEVICE == 'cuda': torch.cuda.empty_cache()

## 5 - Aggregate: mean +/- 1 sigma over seeds

In [ ]:
import pandas as pd
ok = [r for r in rows if r.get('status') == 'ok']
df = pd.DataFrame(ok)
if len(df):
    agg = df.groupby('corpus').agg(
        n=('seed', 'nunique'),
        alpha=('alpha_mean', 'mean'), alpha_sd=('alpha_mean', 'std'),
        theta=('theta_mean', 'mean'), theta_sd=('theta_mean', 'std'),
        theta_cv=('theta_cv', 'mean'), alpha_cv=('alpha_cv', 'mean'),
        tau=('tau_positions', 'mean'), val=('val_loss', 'mean')).round(4)
    print(agg.to_string())
    print(f'\nlearning-rate within-sequence variability is {agg.theta_cv.mean()/max(agg.alpha_cv.mean(),1e-9):.2f}x the gate (mean cv)')
    print('error bars = 1 sigma sample std over seeds (init + batch sampling + AR draw)')
else:
    print('no completed runs yet')

## How to read it

- **alpha (forget gate)** converts to the horizon `tau`; it is corpus-dependent but nearly constant within a
  sequence (`alpha_cv` small).
- **theta (learning rate)** is the token-level control; it spans a wide range across corpora and varies far
  more *within* a sequence (`theta_cv`), which is where test-time adaptation lives.
- Re-run with `DIM in {64, 384, 512}` for the width sweep. Report `mean +/- 1 sigma` over `SEEDS`.
- White noise (`AR_phi0.00`) should sit at the uniform loss (5.545); if it aborts, raise `TOKEN_TARGET`.